# Numerikus módszerek -- 2. gyakorlat
## Gauss-elimináció, LU-felbontás, normák, gépi számábrázolás

Ez a notebook **részletes kidolgozott példákat** és **önálló feladatokat** tartalmaz a következő témákhoz:

1. Gauss-elimináció lépésről lépésre
2. Részleges főelemkiválasztás
3. LU-felbontás gyakorlás
4. Determináns és inverz számítása
5. Vektornormák
6. Mátrixnormák
7. Mátrixok kondíciószáma
8. Önálló feladatok (LER, normák, gépi számábrázolás)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

In [ ]:
def print_augmented(Ab, step_label=""):
    """Kibővített mátrix szép kiírása."""
    n = Ab.shape[0]
    if step_label:
        print(f"--- {step_label} ---")
    for i in range(n):
        row_A = "  ".join(f"{Ab[i,j]:8.4f}" for j in range(n))
        row_b = "  ".join(f"{Ab[i,j]:8.4f}" for j in range(n, Ab.shape[1]))
        print(f"  [{row_A}  | {row_b}]")
    print()

def print_matrix(M, label=""):
    """Mátrix szép kiírása."""
    if label:
        print(label)
    for i in range(M.shape[0]):
        row = "  ".join(f"{M[i,j]:8.4f}" for j in range(M.shape[1]))
        print(f"  [{row}]")
    print()

---
## 1. Gauss-elimináció lépésről lépésre

### Részletes kidolgozott példa

Oldjuk meg a következő lineáris egyenletrendszert Gauss-eliminációval, **részletesen kiírva minden lépést**!

$$
\begin{bmatrix} 3 & -1 & 2 \\ 1 & 2 & -1 \\ 2 & -3 & 5 \end{bmatrix}
\cdot x =
\begin{bmatrix} 7 \\ 0 \\ 11 \end{bmatrix}
$$

**Emlékeztető:** A Gauss-elimináció két fázisból áll:
1. **Előre elimináció:** A kibővített $[A|b]$ mátrixot felső háromszög alakra hozzuk
2. **Visszahelyettesítés:** A felső háromszögű rendszert alulról felfelé megoldjuk

A $k$-adik lépés képlete ($i > k, \ j > k$):
$$
m_{ik} = \frac{a_{ik}^{(k-1)}}{a_{kk}^{(k-1)}}, \qquad a_{ij}^{(k)} = a_{ij}^{(k-1)} - m_{ik} \cdot a_{kj}^{(k-1)}
$$

In [ ]:
A1 = np.array([[3, -1, 2],
               [1,  2, -1],
               [2, -3,  5]], dtype=float)
b1 = np.array([7, 0, 11], dtype=float)

Ab = np.column_stack([A1.copy(), b1.copy()])
print_augmented(Ab, "Kiindulás: [A|b]")

In [ ]:
# === 1. lépés: 1. oszlop kinullázása a főátló alatt ===
Ab = np.column_stack([A1.copy(), b1.copy()])

print("1. lépés: az 1. oszlop elemeit nullázzuk a főátló alatt")
print("="*55)

# 2. sor módosítása
m21 = Ab[1, 0] / Ab[0, 0]
print(f"  m21 = a21/a11 = {A1[1,0]:.0f} / {A1[0,0]:.0f} = {m21:.4f}")
print(f"  2. sor  ←  2. sor - ({m21:.4f}) · 1. sor")
Ab[1] = Ab[1] - m21 * Ab[0]

# 3. sor módosítása
m31 = Ab[2, 0] / Ab[0, 0]
print(f"  m31 = a31/a11 = {A1[2,0]:.0f} / {A1[0,0]:.0f} = {m31:.4f}")
print(f"  3. sor  ←  3. sor - ({m31:.4f}) · 1. sor")
Ab[2] = Ab[2] - m31 * Ab[0]

print()
print_augmented(Ab, "1. lépés után")

In [ ]:
# === 2. lépés: 2. oszlop kinullázása a főátló alatt ===
print("2. lépés: a 2. oszlop elemeit nullázzuk a főátló alatt")
print("="*55)

m32 = Ab[2, 1] / Ab[1, 1]
print(f"  m32 = a32^(1)/a22^(1) = {Ab[2,1]:.4f} / {Ab[1,1]:.4f} = {m32:.4f}")
print(f"  3. sor  ←  3. sor - ({m32:.4f}) · 2. sor")
Ab[2] = Ab[2] - m32 * Ab[1]

print()
print_augmented(Ab, "2. lépés után (felső háromszög alak)")

In [ ]:
# === Visszahelyettesítés ===
print("Visszahelyettesítés (alulról felfelé)")
print("="*55)

n = 3
x = np.zeros(n)

# x3
x[2] = Ab[2, 3] / Ab[2, 2]
print(f"  x3 = {Ab[2,3]:.4f} / {Ab[2,2]:.4f} = {x[2]:.4f}")

# x2
x[1] = (Ab[1, 3] - Ab[1, 2] * x[2]) / Ab[1, 1]
print(f"  x2 = ({Ab[1,3]:.4f} - ({Ab[1,2]:.4f})·({x[2]:.4f})) / {Ab[1,1]:.4f} = {x[1]:.4f}")

# x1
x[0] = (Ab[0, 3] - Ab[0, 1] * x[1] - Ab[0, 2] * x[2]) / Ab[0, 0]
print(f"  x1 = ({Ab[0,3]:.4f} - ({Ab[0,1]:.4f})·({x[1]:.4f}) - ({Ab[0,2]:.4f})·({x[2]:.4f})) / {Ab[0,0]:.4f} = {x[0]:.4f}")

print(f"\nMegoldás: x = {x}")
print(f"Ellenőrzés: A @ x = b? {np.allclose(A1 @ x, b1)}")

In [ ]:
# Visszaellenőrzés soronként
print("Ellenőrzés soronként:")
for i in range(3):
    tagok = " + ".join(f"({A1[i,j]:.0f})·({x[j]:.0f})" for j in range(3))
    ertek = A1[i] @ x
    print(f"  {tagok} = {ertek:.0f}  (elvárt: {b1[i]:.0f})  {'✓' if np.isclose(ertek, b1[i]) else '✗'}")

---
## 2. Részleges főelemkiválasztás

### Miért szükséges?

A Gauss-elimináció $k$-adik lépésében a pivot elem $a_{kk}^{(k-1)}$ lehet **nulla** vagy **nagyon kicsi**:

| Eset | Probléma |
|------|----------|
| Pivot = 0 | Nullával osztás, az algoritmus megáll |
| Pivot $\approx$ 0 | A szorzók ($m_{ik}$) nagyon nagyok, a kerekítési hiba felerősödik |

**Megoldás:** Részleges főelemkiválasztás -- a $k$-adik lépés előtt megkeressük a $k$-adik oszlopban (a $k$-adik sortól lefelé) az abszolút értékben legnagyobb elemet, és kicseréljük a $k$-adik sorral.

### Példa: mikor számít a pivotálás?

$$
\begin{bmatrix} 0.0001 & 1 \\ 1 & 1 \end{bmatrix}
\cdot x =
\begin{bmatrix} 1 \\ 2 \end{bmatrix}
$$

Ha a pivot nagyon kicsi ($\varepsilon = 0.0001$), a szorzó $m = 1/\varepsilon = 10000$ -- **float32** aritmetikában ez komoly hibát okozhat!

In [ ]:
# Kicsi pivot példa -- float32 vs pivotálás
eps = np.float32(0.0001)

A_kicsi = np.array([[eps, 1], [1, 1]], dtype=np.float32)
b_kicsi = np.array([1, 2], dtype=np.float32)

# Pontos megoldás (float64)
x_pontos = np.linalg.solve(A_kicsi.astype(np.float64), b_kicsi.astype(np.float64))
print(f"Pontos megoldás (float64): x = [{x_pontos[0]:.6f}, {x_pontos[1]:.6f}]")

# --- Pivotálás NÉLKÜL (float32) ---
Ab = np.column_stack([A_kicsi.copy(), b_kicsi.copy()])
m = Ab[1, 0] / Ab[0, 0]  # m = 1/0.0001 = 10000
print(f"\nPivotálás NÉLKÜL (float32):")
print(f"  Szorzó: m = {float(m):.1f}  (nagyon nagy!)")
Ab[1] = Ab[1] - m * Ab[0]
x2_nopiv = float(Ab[1, 2] / Ab[1, 1])
x1_nopiv = float((Ab[0, 2] - Ab[0, 1] * x2_nopiv) / Ab[0, 0])
print(f"  Eredmény: x = [{x1_nopiv:.6f}, {x2_nopiv:.6f}]")
hiba_nopiv = np.linalg.norm([x1_nopiv - x_pontos[0], x2_nopiv - x_pontos[1]])
print(f"  Hiba: {hiba_nopiv:.6e}")

# --- Pivotálással (float32) ---
Ab2 = np.column_stack([A_kicsi.copy(), b_kicsi.copy()])
Ab2[[0, 1]] = Ab2[[1, 0]]  # sorcsere
m2 = Ab2[1, 0] / Ab2[0, 0]
print(f"\nPivotálással (float32):")
print(f"  Sorcsere után pivot: {float(Ab2[0,0]):.4f}")
print(f"  Szorzó: m = {float(m2):.6f}  (kicsi, jó!)")
Ab2[1] = Ab2[1] - m2 * Ab2[0]
x2_piv = float(Ab2[1, 2] / Ab2[1, 1])
x1_piv = float((Ab2[0, 2] - Ab2[0, 1] * x2_piv) / Ab2[0, 0])
print(f"  Eredmény: x = [{x1_piv:.6f}, {x2_piv:.6f}]")
hiba_piv = np.linalg.norm([x1_piv - x_pontos[0], x2_piv - x_pontos[1]])
print(f"  Hiba: {hiba_piv:.6e}")

print(f"\nA pivotálás {hiba_nopiv/max(hiba_piv, 1e-30):.0f}x pontosabb eredményt adott!")

### Részletes példa: 3x3 rendszer pivotálással

$$
\begin{bmatrix} 1 & 2 & -1 \\ 4 & 3 & 1 \\ 2 & -1 & 3 \end{bmatrix}
\cdot x =
\begin{bmatrix} 2 \\ 1 \\ 5 \end{bmatrix}
$$

Oldjuk meg részleges főelemkiválasztással! Minden lépésben kiírjuk, melyik elem a főelem és milyen sorcserére van szükség.

In [ ]:
def gauss_elimination_pivot(A, b, verbose=False):
    """Gauss-elimináció részleges főelemkiválasztással."""
    n = len(b)
    Ab = np.column_stack([A.astype(float), b.astype(float)])
    
    if verbose:
        print_augmented(Ab, "Kiindulás")
    
    for k in range(n - 1):
        # Főelemkiválasztás
        max_idx = np.argmax(np.abs(Ab[k:n, k])) + k
        if Ab[max_idx, k] == 0:
            raise ValueError("Szinguláris mátrix!")
        
        if verbose:
            col_vals = [f"|{Ab[i,k]:.4f}|={abs(Ab[i,k]):.4f}" for i in range(k, n)]
            print(f"{k+1}. lépés: {k+1}. oszlop vizsgálata: {', '.join(col_vals)}")
            print(f"  → Maximum: {max_idx+1}. sor")
        
        if max_idx != k:
            Ab[[k, max_idx]] = Ab[[max_idx, k]]
            if verbose:
                print(f"  → Sorcsere: {k+1}. sor ↔ {max_idx+1}. sor")
                print_augmented(Ab, "Sorcsere után")
        else:
            if verbose:
                print(f"  → Nincs szükség sorcserére")
        
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= m * Ab[k, k:]
            if verbose:
                print(f"  m{i+1}{k+1} = {m:.4f}, {i+1}. sor ← {i+1}. sor - ({m:.4f})·{k+1}. sor")
        
        if verbose:
            print()
            print_augmented(Ab, f"{k+1}. lépés után")
    
    # Visszahelyettesítés
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, n] - np.dot(Ab[i, i+1:n], x[i+1:n])) / Ab[i, i]
    return x

A2 = np.array([[1, 2, -1],
               [4, 3,  1],
               [2, -1, 3]], dtype=float)
b2 = np.array([2, 1, 5], dtype=float)

x2 = gauss_elimination_pivot(A2, b2, verbose=True)
print(f"Megoldás: x = {x2}")
print(f"Ellenőrzés: A @ x = b? {np.allclose(A2 @ x2, b2)}")

---
## 3. LU-felbontás gyakorlás

### Emlékeztető

Az $A = LU$ felbontásban:
- $L \in \mathcal{L}_1$: egységátlójú alsó háromszögmátrix (a főátlóban csupa 1)
- $U \in \mathcal{U}$: felső háromszögmátrix

Az $L$ mátrix elemei a Gauss-elimináció **hányadosai** ($m_{ik}$), az $U$ mátrix az elimináció **végeredménye** (a felső háromszög alak).

### Részletes kidolgozott példa

Készítsük el a következő mátrix LU-felbontását **kézzel**, lépésről lépésre!

$$
A = \begin{bmatrix} 4 & -2 & 1 \\ -2 & 4 & -2 \\ 1 & -2 & 4 \end{bmatrix}
$$

In [ ]:
A3 = np.array([[ 4, -2,  1],
               [-2,  4, -2],
               [ 1, -2,  4]], dtype=float)

print_matrix(A3, "A =")

# === 1. lépés: 1. oszlop eliminációja ===
print("=== 1. lépés: 1. oszlop eliminációja ===")
m21 = A3[1, 0] / A3[0, 0]
m31 = A3[2, 0] / A3[0, 0]
print(f"  m21 = a21/a11 = {A3[1,0]:.0f} / {A3[0,0]:.0f} = {m21:.4f}")
print(f"  m31 = a31/a11 = {A3[2,0]:.0f} / {A3[0,0]:.0f} = {m31:.4f}")

U = A3.copy()
U[1] = U[1] - m21 * U[0]
U[2] = U[2] - m31 * U[0]
print_matrix(U, "\nA^(1) =")

# === 2. lépés: 2. oszlop eliminációja ===
print("=== 2. lépés: 2. oszlop eliminációja ===")
m32 = U[2, 1] / U[1, 1]
print(f"  m32 = a32^(1)/a22^(1) = {U[2,1]:.4f} / {U[1,1]:.4f} = {m32:.4f}")

U[2] = U[2] - m32 * U[1]
print_matrix(U, "\nU = A^(2) =")

# === L mátrix összerakása ===
L = np.eye(3)
L[1, 0] = m21
L[2, 0] = m31
L[2, 1] = m32
print_matrix(L, "L =")

print(f"Ellenőrzés: L @ U =")
print_matrix(L @ U)
print(f"L @ U = A? {np.allclose(L @ U, A3)}")

### LER megoldása LU-felbontással

Ha már van $A = LU$ felbontásunk, az $Ax = b$ rendszert két háromszögű rendszerre bontjuk:

1. $Ly = b$ (**előre helyettesítés**) $\rightarrow \mathcal{O}(n^2)$
2. $Ux = y$ (**visszahelyettesítés**) $\rightarrow \mathcal{O}(n^2)$

Oldjuk meg a fenti mátrixszal a $b = [5, -8, 13]^\top$ jobboldalt!

In [ ]:
b3 = np.array([5, -8, 13], dtype=float)

# === 1. lépés: Ly = b (előre helyettesítés) ===
print("1. fázis: Ly = b (előre helyettesítés)")
print("="*50)
y = np.zeros(3)

y[0] = b3[0] / L[0, 0]
print(f"  y1 = b1 / l11 = {b3[0]:.1f} / {L[0,0]:.1f} = {y[0]:.4f}")

y[1] = (b3[1] - L[1, 0] * y[0]) / L[1, 1]
print(f"  y2 = (b2 - l21·y1) / l22 = ({b3[1]:.1f} - ({L[1,0]:.4f})·({y[0]:.4f})) / {L[1,1]:.1f} = {y[1]:.4f}")

y[2] = (b3[2] - L[2, 0] * y[0] - L[2, 1] * y[1]) / L[2, 2]
print(f"  y3 = (b3 - l31·y1 - l32·y2) / l33 = ({b3[2]:.1f} - ({L[2,0]:.4f})·({y[0]:.4f}) - ({L[2,1]:.4f})·({y[1]:.4f})) / {L[2,2]:.1f} = {y[2]:.4f}")

print(f"\n  y = {y}")

# === 2. lépés: Ux = y (visszahelyettesítés) ===
print(f"\n2. fázis: Ux = y (visszahelyettesítés)")
print("="*50)
x3 = np.zeros(3)

x3[2] = y[2] / U[2, 2]
print(f"  x3 = y3 / u33 = {y[2]:.4f} / {U[2,2]:.4f} = {x3[2]:.4f}")

x3[1] = (y[1] - U[1, 2] * x3[2]) / U[1, 1]
print(f"  x2 = (y2 - u23·x3) / u22 = ({y[1]:.4f} - ({U[1,2]:.4f})·({x3[2]:.4f})) / {U[1,1]:.4f} = {x3[1]:.4f}")

x3[0] = (y[0] - U[0, 1] * x3[1] - U[0, 2] * x3[2]) / U[0, 0]
print(f"  x1 = (y1 - u12·x2 - u13·x3) / u11 = ({y[0]:.4f} - ({U[0,1]:.4f})·({x3[1]:.4f}) - ({U[0,2]:.4f})·({x3[2]:.4f})) / {U[0,0]:.4f} = {x3[0]:.4f}")

print(f"\nMegoldás: x = {x3}")
print(f"Ellenőrzés: A @ x = {A3 @ x3}")
print(f"Helyes? {np.allclose(A3 @ x3, b3)}")

In [ ]:
# Összehasonlítás scipy-val
from scipy.linalg import lu

P, L_sp, U_sp = lu(A3)
print("scipy PLU-felbontás:")
print_matrix(P, "P =")
print_matrix(L_sp, "L =")
print_matrix(U_sp, "U =")
print(f"P @ L @ U = A? {np.allclose(P @ L_sp @ U_sp, A3)}")
print(f"A mi L mátrixunk egyezik a scipy L-lel? {np.allclose(L, L_sp)}")

---
## 4. Determináns és inverz számítása

### Determináns GE/LU segítségével

A Gauss-elimináció során kapott $U$ felső háromszögmátrix főátlójának szorzata adja a determinánst:

$$
\det(A) = (-1)^s \cdot \prod_{k=1}^n u_{kk}
$$

ahol $s$ a sorcserék száma (LU-felbontásnál $s = 0$).

In [ ]:
# Determináns az U főátlójából
diag_U = np.diag(U)
det_from_U = np.prod(diag_U)
det_numpy = np.linalg.det(A3)

print(f"U főátlója: {diag_U}")
print(f"det(A) = {diag_U[0]:.4f} · {diag_U[1]:.4f} · {diag_U[2]:.4f} = {det_from_U:.4f}")
print(f"numpy det(A) = {det_numpy:.4f}")
print(f"Egyeznek? {np.isclose(det_from_U, det_numpy)}")

### Inverz mátrix számítása LU-felbontással

Az $A^{-1}$-et úgy számítjuk, hogy az $Ax_j = e_j$ rendszert megoldjuk minden $j = 1, \ldots, n$-re, ahol $e_j$ a $j$-edik egységvektor.

Ha az LU-felbontás rendelkezésünkre áll, csak előre- és visszahelyettesítést kell végeznünk ($n$-szer), ami összesen $\mathcal{O}(n^2 \cdot n) = \mathcal{O}(n^3)$ művelet.

In [ ]:
def forward_sub(L, b):
    """Ly = b megoldása előre helyettesítéssel."""
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = (b[i] - np.dot(L[i, :i], y[:i])) / L[i, i]
    return y

def backward_sub(U, y):
    """Ux = y megoldása visszahelyettesítéssel."""
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - np.dot(U[i, i+1:], x[i+1:])) / U[i, i]
    return x

# Inverz számítása oszloponként
n = 3
A_inv = np.zeros((n, n))

print("Inverz számítása LU-felbontással:")
for j in range(n):
    ej = np.zeros(n)
    ej[j] = 1.0
    y_j = forward_sub(L, ej)
    x_j = backward_sub(U, y_j)
    A_inv[:, j] = x_j
    print(f"  {j+1}. oszlop (e_{j+1} jobboldallal): x = [{', '.join(f'{v:.4f}' for v in x_j)}]")

print_matrix(A_inv, f"\nA^(-1) =")

print(f"Ellenőrzés: A @ A^(-1) ≈ I? {np.allclose(A3 @ A_inv, np.eye(3))}")
print_matrix(np.linalg.inv(A3), "\nnumpy A^(-1) =")

---
## 5. Vektornormák

### Összefoglaló

- $\lVert x \rVert_1 = \sum_{i=1}^n |x_i|$ — **Manhattan-norma**
- $\lVert x \rVert_2 = \left(\sum_{i=1}^n |x_i|^2\right)^{1/2}$ — **Euklideszi norma**
- $\lVert x \rVert_\infty = \max_i |x_i|$ — **Csebisev-norma**
- $\lVert x \rVert_p = \left(\sum_{i=1}^n |x_i|^p\right)^{1/p}$ — **$p$-norma** ($p \ge 1$)

### Normák közötti egyenlőtlenségek

$$
\lVert x \rVert_\infty \le \lVert x \rVert_2 \le \lVert x \rVert_1 \le \sqrt{n} \cdot \lVert x \rVert_2 \le n \cdot \lVert x \rVert_\infty
$$

### Részletes kidolgozott példa

Számítsuk ki a következő vektorok normáit **kézzel**, majd ellenőrizzük Pythonnal!

$$
a = \begin{bmatrix} -3 \\ 4 \\ 0 \\ -1 \end{bmatrix}, \qquad
c = \begin{bmatrix} 2 \\ -2 \\ 2 \\ -2 \end{bmatrix}
$$

In [ ]:
a = np.array([-3, 4, 0, -1], dtype=float)
c_vec = np.array([2, -2, 2, -2], dtype=float)

for vec, name in [(a, 'a'), (c_vec, 'c')]:
    n = len(vec)
    print(f"{'='*55}")
    print(f"{name} = {vec}")
    print(f"{'='*55}")
    
    # 1-norma
    n1 = np.sum(np.abs(vec))
    tagok = ' + '.join(f'|{v:.0f}|' for v in vec)
    print(f"  ||{name}||_1   = {tagok} = {n1:.1f}")
    
    # 2-norma
    n2 = np.sqrt(np.sum(vec**2))
    tagok = ' + '.join(f'{v:.0f}²' for v in vec)
    print(f"  ||{name}||_2   = sqrt({tagok}) = sqrt({np.sum(vec**2):.0f}) = {n2:.4f}")
    
    # inf-norma
    ninf = np.max(np.abs(vec))
    tagok = ', '.join(f'|{v:.0f}|' for v in vec)
    print(f"  ||{name}||_inf = max({tagok}) = {ninf:.1f}")
    
    # 3-norma
    p = 3
    np3 = np.sum(np.abs(vec)**p)**(1/p)
    tagok = ' + '.join(f'|{v:.0f}|³' for v in vec)
    print(f"  ||{name}||_3   = ({tagok})^(1/3) = ({np.sum(np.abs(vec)**3):.0f})^(1/3) = {np3:.4f}")
    
    # Egyenlőtlenségek
    print(f"\n  Egyenlőtlenségek ellenőrzése:")
    print(f"    ||{name}||_inf = {ninf:.2f}  ≤  ||{name}||_2 = {n2:.2f}  ≤  ||{name}||_1 = {n1:.2f}  ≤  sqrt(n)·||{name}||_2 = {np.sqrt(n)*n2:.2f}  ≤  n·||{name}||_inf = {n*ninf:.2f}")
    ok = (ninf <= n2 + 1e-10 and n2 <= n1 + 1e-10 and 
          n1 <= np.sqrt(n)*n2 + 1e-10 and np.sqrt(n)*n2 <= n*ninf + 1e-10)
    print(f"    Mind teljesül? {ok}")
    
    # numpy ellenőrzés
    print(f"\n  numpy ellenőrzés: ||{name}||_1={np.linalg.norm(vec,1):.4f}, ||{name}||_2={np.linalg.norm(vec,2):.4f}, ||{name}||_inf={np.linalg.norm(vec,np.inf):.4f}")
    print()

In [ ]:
# p-norma viselkedése p → ∞ esetén
x_demo = np.array([-3, 4, 0, -1], dtype=float)
p_values = [1, 1.5, 2, 3, 5, 10, 20, 50, 100]

print(f"x = {x_demo}")
print(f"||x||_inf = {np.linalg.norm(x_demo, np.inf):.4f}\n")

print(f"{'p':>6} {'||x||_p':>12}")
print("-" * 20)
for p in p_values:
    print(f"{p:>6} {np.linalg.norm(x_demo, p):>12.4f}")
print(f"{'inf':>6} {np.linalg.norm(x_demo, np.inf):>12.4f}")
print("\n→ p → ∞ esetén ||x||_p → ||x||_inf = max|x_i|")

---
## 6. Mátrixnormák

### Összefoglaló

- $\lVert A \rVert_F = \left(\sum_{i,j} |a_{ij}|^2\right)^{1/2}$ — **Frobenius-norma** (NEM indukált!)
- $\lVert A \rVert_1 = \max_j \sum_i |a_{ij}|$ — **Oszlopnorma** (indukált)
- $\lVert A \rVert_\infty = \max_i \sum_j |a_{ij}|$ — **Sornorma** (indukált)
- $\lVert A \rVert_2 = \sqrt{\max_i \lambda_i(A^\top A)}$ — **Spektrálnorma** (indukált)

**Fontos:** Indukált normákra $\lVert I \rVert = 1$, de $\lVert I \rVert_F = \sqrt{n} \neq 1$.

### Részletes kidolgozott példa

Számítsuk ki az alábbi mátrix minden normáját!

$$
A = \begin{bmatrix} 2 & -1 & 0 \\ -1 & 2 & -1 \\ 0 & -1 & 2 \end{bmatrix}
$$

In [ ]:
A6 = np.array([[ 2, -1,  0],
               [-1,  2, -1],
               [ 0, -1,  2]], dtype=float)

print_matrix(A6, "A =")

# --- Frobenius-norma ---
elems_sq = A6**2
nz_elems = [(i,j, A6[i,j]) for i in range(3) for j in range(3) if A6[i,j] != 0]
tagok = ' + '.join(f'{A6[i,j]:.0f}²' for i,j,_ in nz_elems)
fro = np.sqrt(np.sum(elems_sq))
print(f"||A||_F = sqrt({tagok}) = sqrt({np.sum(elems_sq):.0f}) = {fro:.4f}")

# --- 1-norma (max oszlopösszeg) ---
print(f"\n||A||_1 (oszlopnorma = max oszlopösszeg):")
col_sums = np.sum(np.abs(A6), axis=0)
for j in range(3):
    tagok = ' + '.join(f'|{A6[i,j]:.0f}|' for i in range(3))
    print(f"  {j+1}. oszlop: {tagok} = {col_sums[j]:.0f}")
print(f"  → max = {np.max(col_sums):.0f}")

# --- inf-norma (max sorösszeg) ---
print(f"\n||A||_inf (sornorma = max sorösszeg):")
row_sums = np.sum(np.abs(A6), axis=1)
for i in range(3):
    tagok = ' + '.join(f'|{A6[i,j]:.0f}|' for j in range(3))
    print(f"  {i+1}. sor: {tagok} = {row_sums[i]:.0f}")
print(f"  → max = {np.max(row_sums):.0f}")

# --- 2-norma (spektrálnorma) ---
print(f"\n||A||_2 (spektrálnorma):")
ATA = A6.T @ A6
eigenvalues_ATA = np.linalg.eigvalsh(ATA)
norm2 = np.sqrt(np.max(eigenvalues_ATA))
print(f"  A^T·A sajátértékei: {eigenvalues_ATA}")
print(f"  ||A||_2 = sqrt(max λ(A^T·A)) = sqrt({np.max(eigenvalues_ATA):.4f}) = {norm2:.4f}")

# Szimmetrikus eset: ||A||_2 = ρ(A)
eig_A = np.linalg.eigvalsh(A6)
rho_A = np.max(np.abs(eig_A))
print(f"\n  A szimmetrikus → ||A||_2 = ρ(A)")
print(f"  A sajátértékei: {eig_A}")
print(f"  ρ(A) = max|λ_i| = {rho_A:.4f}")
print(f"  ||A||_2 = ρ(A)? {np.isclose(norm2, rho_A)}")

# numpy ellenőrzés
print(f"\nnumpy ellenőrzés:")
print(f"  ||A||_F   = {np.linalg.norm(A6, 'fro'):.4f}")
print(f"  ||A||_1   = {np.linalg.norm(A6, 1):.4f}")
print(f"  ||A||_inf = {np.linalg.norm(A6, np.inf):.4f}")
print(f"  ||A||_2   = {np.linalg.norm(A6, 2):.4f}")

In [ ]:
# Szubmultiplikativitás ellenőrzése véletlenszerű mátrixokkal
np.random.seed(42)
A_rand = np.random.randn(3, 3)
B_rand = np.random.randn(3, 3)
AB = A_rand @ B_rand

print("Szubmultiplikativitás: ||A·B|| ≤ ||A||·||B||")
print(f"{'Norma':>12} {'||AB||':>10} {'||A||·||B||':>12} {'Teljesül?':>10}")
print("-" * 48)
for name, p in [('Frobenius', 'fro'), ('1-norma', 1), ('inf-norma', np.inf), ('2-norma', 2)]:
    nAB = np.linalg.norm(AB, p)
    nA = np.linalg.norm(A_rand, p)
    nB = np.linalg.norm(B_rand, p)
    print(f"{name:>12} {nAB:>10.4f} {nA*nB:>12.4f} {str(nAB <= nA*nB + 1e-10):>10}")

---
## 7. Mátrixok kondíciószáma

### Definíció

Az $A$ mátrix **kondíciószáma** egy adott mátrixnormában:

$$
\kappa(A) = \lVert A \rVert \cdot \lVert A^{-1} \rVert
$$

### Tétel: a megoldás relatív hibájának korlátja

Ha $A(x + \Delta x) = b + \Delta b$, akkor:

$$
\frac{\lVert \Delta x \rVert}{\lVert x \rVert} \le \kappa(A) \cdot \frac{\lVert \Delta b \rVert}{\lVert b \rVert}
$$

| $\kappa(A)$ | Kondicionáltság |
|-------------|------------------|
| $\approx 1$ | Jól kondicionált |
| $\approx 10^k$ | A megoldásban kb. $k$ tizedesjegyet veszíthetünk |
| $\gg 1$ | Rosszul kondicionált |

In [ ]:
# Jól vs. rosszul kondicionált mátrixok összehasonlítása

# 1) Forgatás mátrix (ortogonális → κ = 1)
theta = np.pi / 6
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

# 2) Hilbert-mátrix 3x3 (közepesen rossz)
H3 = np.array([[1, 1/2, 1/3],
               [1/2, 1/3, 1/4],
               [1/3, 1/4, 1/5]])

# 3) Hilbert-mátrix 5x5 (nagyon rossz)
H5 = np.array([[1/(i+j+1) for j in range(5)] for i in range(5)])

print("=== Kondíciószámok összehasonlítása ===")
print(f"{'Mátrix':>22} {'κ_2':>14} {'log10(κ)':>10} {'Értékelés':>20}")
print("-" * 70)
for name, M in [("Forgatás (2x2)", Q), ("Hilbert 3x3", H3), ("Hilbert 5x5", H5)]:
    k2 = np.linalg.cond(M, 2)
    logk = np.log10(k2)
    if logk < 1:
        ertekeles = "Kiváló"
    elif logk < 4:
        ertekeles = "Elfogadható"
    else:
        ertekeles = f"~{logk:.0f} jegy vész el!"
    print(f"{name:>22} {k2:>14.2f} {logk:>10.1f} {ertekeles:>20}")

### Szemléltető példa: perturbáció hatása rosszul kondicionált rendszerre

A Hilbert-mátrix rossz kondíciószáma miatt a jobboldal **nagyon kicsi** perturbációjából **nagy** hiba keletkezik a megoldásban.

In [ ]:
# Perturbációs kísérlet: Hilbert 5x5
H = np.array([[1/(i+j+1) for j in range(5)] for i in range(5)])
x_exact = np.ones(5)  # előre megválasztott megoldás
b_exact = H @ x_exact

# Perturbált jobboldal (nagyon kicsi!)
np.random.seed(0)
delta_b = np.random.randn(5) * 1e-8
b_pert = b_exact + delta_b
x_pert = np.linalg.solve(H, b_pert)

delta_x = x_pert - x_exact
rel_hiba_b = np.linalg.norm(delta_b) / np.linalg.norm(b_exact)
rel_hiba_x = np.linalg.norm(delta_x) / np.linalg.norm(x_exact)
kappa = np.linalg.cond(H, 2)

print(f"Hilbert-mátrix 5×5 kondíciószáma: κ_2 = {kappa:.2e}")
print(f"\nJobboldal relatív perturbációja:  ||Δb||/||b|| = {rel_hiba_b:.2e}")
print(f"Megoldás relatív hibája:          ||Δx||/||x|| = {rel_hiba_x:.2e}")
print(f"Felerősítés:                       {rel_hiba_x / rel_hiba_b:.1f}×")
print(f"\nElméleti korlát: κ · (rel. hiba b) = {kappa * rel_hiba_b:.2e}")
print(f"Tényleges relatív hiba:              {rel_hiba_x:.2e}")
print(f"Korlát teljesül? {rel_hiba_x <= kappa * rel_hiba_b + 1e-10}")
print(f"\nPontos megoldás:     x = {x_exact}")
print(f"Perturbált megoldás: x = {np.array2string(x_pert, precision=4)}")

In [ ]:
# Hilbert-mátrixok kondíciószámának növekedése
print(f"{'n':>4} {'κ_2(H_n)':>18} {'log10(κ)':>14}")
print("-" * 40)
for n in range(2, 12):
    Hn = np.array([[1/(i+j+1) for j in range(n)] for i in range(n)])
    kn = np.linalg.cond(Hn, 2)
    print(f"{n:>4} {kn:>18.2e} {np.log10(kn):>14.1f}")

print("\nA Hilbert-mátrixok kondíciószáma exponenciálisan nő!")
print("n=10 esetén már ~10^13: a 16 jegyes double aritmetikában")
print("alig 3 tizedesjegy megbízható a megoldásban.")

---
## 8. Önálló feladatok

Az alábbi feladatokat a fenti példák mintájára oldják meg! Először próbálják meg **kézzel** (papír-ceruza), majd ellenőrizzék Pythonnal!

### 1. feladat: Gauss-elimináció

Oldjuk meg Gauss-eliminációval (pivotálás nélkül):

$$
\begin{bmatrix} 2 & 1 & -1 \\ -3 & -1 & 2 \\ -2 & 1 & 2 \end{bmatrix}
\cdot x =
\begin{bmatrix} 8 \\ -11 \\ -3 \end{bmatrix}
$$

*(Elvárt megoldás: $x = [2, 3, -1]^\top$)*

In [ ]:
# 1. feladat -- megoldás ellenőrzése
A_f1 = np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], dtype=float)
b_f1 = np.array([8, -11, -3], dtype=float)

x_f1 = np.linalg.solve(A_f1, b_f1)
print(f"Megoldás: x = {x_f1}")
print(f"Ellenőrzés: A @ x = b? {np.allclose(A_f1 @ x_f1, b_f1)}")

### 2. feladat: LU-felbontás

Készítsük el a következő mátrix LU-felbontását! Írjuk ki az $L$ és $U$ mátrixokat, és ellenőrizzük, hogy $LU = A$!

$$
A = \begin{bmatrix} 3 & -1 & 2 \\ 6 & -1 & 5 \\ -9 & 7 & -2 \end{bmatrix}
$$

In [ ]:
# 2. feladat -- megoldás ellenőrzése
A_f2 = np.array([[3, -1, 2], [6, -1, 5], [-9, 7, -2]], dtype=float)

P_f2, L_f2, U_f2 = lu(A_f2)
print_matrix(L_f2, "L =")
print_matrix(U_f2, "U =")
print(f"P @ L @ U = A? {np.allclose(P_f2 @ L_f2 @ U_f2, A_f2)}")
print(f"P = I (nincs sorcsere)? {np.allclose(P_f2, np.eye(3))}")

### 3. feladat: Pivotálás szükséges

Oldjuk meg részleges főelemkiválasztással:

$$
\begin{bmatrix} 0 & 2 & 1 \\ 3 & -1 & 2 \\ 1 & 4 & -1 \end{bmatrix}
\cdot x =
\begin{bmatrix} 1 \\ 7 \\ 3 \end{bmatrix}
$$

Miért nem működne pivotálás nélkül?

In [ ]:
# 3. feladat -- megoldás ellenőrzése
A_f3 = np.array([[0, 2, 1], [3, -1, 2], [1, 4, -1]], dtype=float)
b_f3 = np.array([1, 7, 3], dtype=float)

print("Miért nem megy pivotálás nélkül?")
print(f"  a11 = {A_f3[0,0]:.0f} → nulla pivot! Nullával kellene osztani.\n")

x_f3 = gauss_elimination_pivot(A_f3, b_f3, verbose=True)
print(f"Megoldás: x = {x_f3}")
print(f"Ellenőrzés: {np.allclose(A_f3 @ x_f3, b_f3)}")

### 4. feladat: Vektornormák és egyenlőtlenségek

Számítsuk ki az alábbi vektor összes normáját ($1, 2, 3, \infty$), és ellenőrizzük a normák közötti egyenlőtlenségeket!

$$
v = \begin{bmatrix} 1 \\ -2 \\ 3 \\ -4 \\ 5 \end{bmatrix}
$$

Melyik egyenlőtlenség lesz a legszorosabb (hol közelíti meg legjobban az egyenlőség)?

In [ ]:
# 4. feladat -- megoldás ellenőrzése
v_f4 = np.array([1, -2, 3, -4, 5], dtype=float)
n = len(v_f4)

print(f"v = {v_f4}\n")
for p_name, p in [("1", 1), ("2", 2), ("3", 3), ("inf", np.inf)]:
    print(f"  ||v||_{p_name:>3} = {np.linalg.norm(v_f4, p):.4f}")

n1 = np.linalg.norm(v_f4, 1)
n2 = np.linalg.norm(v_f4, 2)
ninf = np.linalg.norm(v_f4, np.inf)
print(f"\nEgyenlőtlenségek:")
print(f"  ||v||_inf={ninf:.2f} ≤ ||v||_2={n2:.2f} ≤ ||v||_1={n1:.2f} ≤ sqrt(n)·||v||_2={np.sqrt(n)*n2:.2f} ≤ n·||v||_inf={n*ninf:.2f}")

### 5. feladat: Mátrixnormák és kondíciószám

Számítsuk ki az alábbi mátrix Frobenius-, $1$-, $\infty$- és $2$-normáját, majd a kondíciószámot $\kappa_2(A)$-t!

$$
A = \begin{bmatrix} 10 & -1 & 0 \\ -1 & 10 & -1 \\ 0 & -1 & 10 \end{bmatrix}
$$

Ez a mátrix jól vagy rosszul kondicionált? Miért?

In [ ]:
# 5. feladat -- megoldás ellenőrzése
A_f5 = np.array([[10, -1, 0], [-1, 10, -1], [0, -1, 10]], dtype=float)

print(f"||A||_F   = {np.linalg.norm(A_f5, 'fro'):.4f}")
print(f"||A||_1   = {np.linalg.norm(A_f5, 1):.4f}")
print(f"||A||_inf = {np.linalg.norm(A_f5, np.inf):.4f}")
print(f"||A||_2   = {np.linalg.norm(A_f5, 2):.4f}")
print(f"kappa_2(A)    = {np.linalg.cond(A_f5, 2):.4f}")
kappa_val = np.linalg.cond(A_f5, 2)
print(f"\nkappa ~ {kappa_val:.1f} --> jol kondicionalt")
print("(diagonalisan dominans: |a_ii| > sum_j |a_ij| (j!=i) minden sorban)")

### 6. feladat: Kondíciószám és a megoldás pontossága

Az alábbi két mátrix közül melyikkel kapjuk a megbízhatóbb megoldást? Indokoljunk a kondíciószám segítségével!

$$
A_1 = \begin{bmatrix} 1 & 2 \\ 1.0001 & 2 \end{bmatrix}, \qquad
A_2 = \begin{bmatrix} 3 & 1 \\ 1 & 3 \end{bmatrix}
$$

Oldjuk meg mindkét rendszert $b = [3, 3.0001]^\top$ jobboldallal, majd perturbáljuk $b$-t $10^{-4}$-nel és vizsgáljuk a megoldás változását!

In [ ]:
# 6. feladat -- megoldás ellenőrzése
A_f6a = np.array([[1, 2], [1.0001, 2]], dtype=float)
A_f6b = np.array([[3, 1], [1, 3]], dtype=float)
b_f6 = np.array([3, 3.0001], dtype=float)

for name, M in [("A1 (majdnem szinguláris)", A_f6a), ("A2 (jól kondicionált)", A_f6b)]:
    x_orig = np.linalg.solve(M, b_f6)
    delta = np.array([1e-4, 0])
    x_pert = np.linalg.solve(M, b_f6 + delta)
    kappa = np.linalg.cond(M, 2)
    print(f"{name}:")
    print(f"  κ_2       = {kappa:.2e}")
    print(f"  x         = {x_orig}")
    print(f"  x_pert    = {x_pert}")
    print(f"  ||Δx||/||x|| = {np.linalg.norm(x_pert - x_orig) / np.linalg.norm(x_orig):.2e}")
    print()

---
## Összefoglalás

| Téma | Lényeg |
|------|--------|
| **Gauss-elimináció** | Kibővített mátrix $[A{\mid}b]$, előre elimináció + visszahelyettesítés |
| **Főelemkiválasztás** | Kicsi/nulla pivot elkerülése, numerikus stabilitás |
| **LU-felbontás** | $A = LU$, $L$ elemei a GE hányadosok, $U$ a felső háromszög |
| **Determináns** | $\det(A) = (-1)^s \prod u_{kk}$ |
| **Inverz** | $n$ darab $Ax_j = e_j$ megoldása LU-val |
| **Vektornormák** | $1, 2, \infty, p$-normák, egyenlőtlenségek |
| **Mátrixnormák** | Frobenius (nem indukált), $1, 2, \infty$ (indukált) |
| **Kondíciószám** | $\kappa(A) = \lVert A \rVert \cdot \lVert A^{-1} \rVert$, nagy $\kappa$ = rossz kondíció |
| **Gépi számhalmaz** | $M(t, k^-, k^+)$: $\varepsilon_0, \varepsilon_1, M_\infty$, elemek felsorolása |
| **Input hiba** | $\delta_{fl} \le \frac{1}{2}\varepsilon_1 = 2^{-t}$, túl-/alulcsordulás |
| **Hibaterjedés** | Kivonás veszélyes (kioltás), konjugált alak segít |

---
## Összefoglalás

| Téma | Lényeg |
|------|--------|
| **Gauss-elimináció** | Kibővített mátrix $[A{\mid}b]$, előre elimináció + visszahelyettesítés |
| **Főelemkiválasztás** | Kicsi/nulla pivot elkerülése, numerikus stabilitás |
| **LU-felbontás** | $A = LU$, $L$ elemei a GE hányadosok, $U$ a felső háromszög |
| **Determináns** | $\det(A) = (-1)^s \prod u_{kk}$ |
| **Inverz** | $n$ darab $Ax_j = e_j$ megoldása LU-val |
| **Vektornormák** | $1, 2, \infty, p$-normák, egyenlőtlenségek |
| **Mátrixnormák** | Frobenius (nem indukált), $1, 2, \infty$ (indukált) |
| **Kondíciószám** | $\kappa(A) = \lVert A \rVert \cdot \lVert A^{-1} \rVert$, nagy $\kappa$ = rossz kondíció |

In [ ]:
# 7. feladat -- megoldás ellenőrzése
t, k_min, k_max = 3, -1, 2

# a) Nevezetes értékek
eps0 = 2**(k_min - 1)
eps1 = 2**(1 - t)
M_inf = (1 - 2**(-t)) * 2**k_max
M_db = 2 * 2**(t - 1) * (k_max - k_min + 1) + 1

print(f"M({t}, {k_min}, {k_max}) nevezetes értékei:")
print(f"  eps_0 = 2^({k_min}-1) = 2^({k_min-1}) = {eps0}")
print(f"  eps_1 = 2^(1-{t}) = 2^({1-t}) = {eps1}")
print(f"  M_inf = (1 - 2^(-{t})) * 2^{k_max} = {1 - 2**(-t):.4f} * {2**k_max} = {M_inf}")
print(f"  |M|   = 2 * 2^({t}-1) * ({k_max}-({k_min})+1) + 1 = 2*{2**(t-1)}*{k_max-k_min+1}+1 = {M_db}")

# b) Pozitív elemek felsorolása
print(f"\nPozitív elemek:")
print(f"  Mantisszák (0.1m2m3): 0.100=1/2, 0.101=5/8, 0.110=6/8=3/4, 0.111=7/8")
mantisszak = [0.5, 5/8, 3/4, 7/8]
m_nevek = ["0.100 = 1/2", "0.101 = 5/8", "0.110 = 3/4", "0.111 = 7/8"]

pozitiv = []
print(f"\n  {'k':>3}  {'m=1/2':>8}  {'m=5/8':>8}  {'m=3/4':>8}  {'m=7/8':>8}")
print("  " + "-" * 40)
for k in range(k_min, k_max + 1):
    vals = [m * 2**k for m in mantisszak]
    pozitiv.extend(vals)
    print(f"  {k:>3}  {vals[0]:>8.4f}  {vals[1]:>8.4f}  {vals[2]:>8.4f}  {vals[3]:>8.4f}")

print(f"\n  Pozitív elemek száma: {len(pozitiv)}")
print(f"  Összes elem (+ negatívok + 0): {2*len(pozitiv) + 1} = {M_db}")

# c) Számegyenes
pozitiv_sorted = sorted(pozitiv)
plt.figure(figsize=(12, 2.5))
plt.plot(pozitiv_sorted, [0]*len(pozitiv_sorted), 'ro', markersize=8)
for x in pozitiv_sorted:
    plt.annotate(f'{x:.3f}', (x, 0), textcoords="offset points",
                 xytext=(0, 10), ha='center', fontsize=6, rotation=60)
plt.axhline(y=0, color='gray', linewidth=0.5)
plt.xlabel('Ertek')
plt.title(f'$M({t}, {k_min}, {k_max})$ pozitiv gepi szamok')
plt.yticks([])
plt.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("A szamok NEM egyenletesen helyezkednek el:")
print("kisebb szamoknal surubben, nagyobbanal ritkabban.")

### 8. feladat: Input hiba és kerekítés

Adott az $M(3, -1, 2)$ gépi számhalmaz (az előző feladat).

**a)** Határozzuk meg a következő valós számok képét az $fl$ input függvény szerint, azaz keressük meg a legközelebbi gépi számot! Számítsuk ki az abszolút és a relatív hibát!

| $x$ | $fl(x)$ | abszolút hiba | relatív hiba |
|-----|---------|---------------|--------------|
| $0.7$ | ? | ? | ? |
| $1.1$ | ? | ? | ? |
| $3.0$ | ? | ? | ? |

**b)** Ellenőrizzük, hogy minden esetben teljesül-e az input hiba tétele:

$$\frac{|x - fl(x)|}{|x|} \le \frac{1}{2} \varepsilon_1 = 2^{-t} = 2^{-3} = 0.125$$

**c)** Mi történik, ha $x = 4.0$? (Tipp: $M_\infty = 3.5$, azaz $x > M_\infty$!)

In [ ]:
# 8. feladat -- megoldás ellenőrzése
t, k_min, k_max = 3, -1, 2
eps1 = 2**(1 - t)
M_inf = (1 - 2**(-t)) * 2**k_max

# Az M(3,-1,2) összes pozitív eleme (rendezve)
mantisszak = [0.5, 5/8, 3/4, 7/8]
gepi_szamok = sorted(set(
    m * 2**k for m in mantisszak for k in range(k_min, k_max + 1)
))

print(f"M(3,-1,2) pozitiv elemei: {[f'{x:.4f}' for x in gepi_szamok]}")
print(f"eps_1 = {eps1}, eps_1/2 = {eps1/2} = {2**(-t)}")
print()

x_values = [0.7, 1.1, 3.0]

print(f"{'x':>6}  {'fl(x)':>8}  {'|x-fl(x)|':>10}  {'rel.hiba':>10}  {'<= eps1/2?':>10}")
print("-" * 52)
for x in x_values:
    # Legközelebbi gépi szám keresése
    idx = np.argmin([abs(x - g) for g in gepi_szamok])
    fl_x = gepi_szamok[idx]
    abs_hiba = abs(x - fl_x)
    rel_hiba = abs_hiba / abs(x)
    ok = rel_hiba <= eps1 / 2 + 1e-15
    print(f"{x:>6.1f}  {fl_x:>8.4f}  {abs_hiba:>10.4f}  {rel_hiba:>10.4f}  {'IGEN' if ok else 'NEM':>10}")

# c) x = 4.0 > M_inf
print(f"\nc) x = 4.0: M_inf = {M_inf}, tehat 4.0 > M_inf!")
print(f"   Ez tulcsordulas (overflow): a szam nem abrazolhato az M(3,-1,2)-ben.")

### 9. feladat: Hibaterjedés és katasztrofális kioltás

**a)** Egy kísérletben megmértük az alábbi értékeket:

$$a = 1.0 \pm 0.01, \quad b = 0.998 \pm 0.01$$

Számítsuk ki az $a - b$ különbség **abszolút** és **relatív** hibakorlátját!

*Emlékeztető:* $\Delta_{a-b} = \Delta_a + \Delta_b$, $\quad \delta_{a-b} = \frac{|a| \cdot \delta_a + |b| \cdot \delta_b}{|a - b|}$

Miért problémás a kivonás ebben az esetben?

**b)** Számítsuk ki az $f(x) = \sqrt{x+1} - \sqrt{x}$ értékét $x = 10000$ esetén kétféleképpen:

1. Közvetlenül: $\sqrt{10001} - \sqrt{10000}$
2. Konjugált alakkal: $\frac{1}{\sqrt{x+1} + \sqrt{x}}$

Hasonlítsuk össze az eredményeket float32 és float64 aritmetikában!

**c)** A $\sin(\pi)$ értéke analitikusan 0. Mennyi a Pythonban kiszámított `np.sin(np.pi)` értéke, és miért nem nulla?

In [ ]:
# 9. feladat -- megoldás ellenőrzése

# a) Hibaterjedés kivonásnál
a_val, b_val = 1.0, 0.998
Delta_a, Delta_b = 0.01, 0.01
delta_a = Delta_a / abs(a_val)  # relatív hiba
delta_b = Delta_b / abs(b_val)

kul = a_val - b_val
Delta_kul = Delta_a + Delta_b
delta_kul = (abs(a_val) * delta_a + abs(b_val) * delta_b) / abs(kul)

print("a) Hibaterjedés kivonásnál")
print(f"   a = {a_val}, b = {b_val}, a - b = {kul}")
print(f"   Delta_a = {Delta_a}, Delta_b = {Delta_b}")
print(f"   delta_a = {delta_a:.2%}, delta_b = {delta_b:.2%}")
print(f"   Delta_(a-b) = {Delta_a} + {Delta_b} = {Delta_kul}")
print(f"   delta_(a-b) = ({abs(a_val)}*{delta_a:.4f} + {abs(b_val)}*{delta_b:.4f}) / {abs(kul)}")
print(f"              = {delta_kul:.1f} = {delta_kul:.0%}")
print(f"   A bemeneti rel. hiba ~1%, de az eredmeny rel. hibaja {delta_kul:.0%}!")
print(f"   --> Katasztrofalis kioltas: kozel azonos szamok kivonasa.\n")

# b) Konjugált alak
x = 10000

for dtype, nev in [(np.float32, "float32"), (np.float64, "float64")]:
    x_t = dtype(x)
    kozvetlen = dtype(np.sqrt(dtype(x_t + 1)) - np.sqrt(x_t))
    konjugalt = dtype(dtype(1.0) / (np.sqrt(dtype(x_t + 1)) + np.sqrt(x_t)))
    print(f"b) x = {x}, {nev}:")
    print(f"   Kozvetlen:     sqrt({x}+1) - sqrt({x}) = {float(kozvetlen):.10e}")
    print(f"   Konjugalt:     1/(sqrt({x}+1)+sqrt({x})) = {float(konjugalt):.10e}")
    if nev == "float32":
        print(f"   Elteres: {abs(float(kozvetlen) - float(konjugalt)):.6e}")
    print()

# c) sin(pi)
print(f"c) np.sin(np.pi) = {np.sin(np.pi):.20e}")
print(f"   np.pi = {np.pi!r}")
print(f"   np.pi nem pontosan pi, hanem a hozza legkozelebbi float64 szam.")
print(f"   Ezert sin(np.pi) sem pontosan 0, hanem ~1.22e-16 (a gepi epszilon nagyságrendje).")